# Notebook 55 — Cluster-Conditioned Residual-Field Corrections

Notebook 54 showed that classifying regimes and switching coefficient predictors is not enough.
This notebook treats clusters as **residual-field structures** rather than coefficient buckets.

We model

\[
g(r,c)=g_{\text{global}}(r,c;\beta)+\Delta g_{\text{cluster}}(r,c)
\]

where:
- `g_global` is the global coefficient-template field
- `Δg_cluster` is a cluster-conditioned correction field
- cluster identity is predicted from metadata, with optional soft gating

## Main goals

1. Fit a global template coefficient predictor.
2. Compute residual fields relative to that global template.
3. Fit cluster-conditioned residual-field correction models.
4. Compare:
   - global template only
   - hard-gated residual correction
   - soft-gated residual correction
5. Evaluate whether residual corrections repair harder holdout blocks.

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.cluster import KMeans

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

In [ ]:
# Fixed template, coefficients, and latent clusters

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def trajectory_gap_from_field(sub, field_fn_ref, field_fn_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(field_fn, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            g = float(np.clip(field_fn(r, c), -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(field_fn_ref, r0)
        t_cmp = integrate(field_fn_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(["system", "task", "forcing_mode", "k", "flow_mode"]):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)

meta_df = coef_df[["regime_id", "system", "task", "forcing_mode", "flow_mode", "k"]].copy()
X_meta = pd.get_dummies(meta_df[["system", "task", "forcing_mode", "flow_mode"]], drop_first=False)
X_meta["k"] = coef_df["k"].astype(float).values
X_meta["k2"] = coef_df["k"].astype(float).values**2
Y_coef = coef_df[COEF_COLS].copy()

coef_scaler = StandardScaler()
Y_std = coef_scaler.fit_transform(Y_coef.to_numpy(dtype=float))
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(Y_std)
coef_df["z1"] = Z[:, 0]
coef_df["z2"] = Z[:, 1]

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
coef_df["cluster"] = kmeans.fit_predict(Z)

display(coef_df.head())

## Global template coefficient predictor

In [ ]:
def fit_global_beta_predictor(X_train, Y_train):
    model = LinearRegression()
    model.fit(np.asarray(X_train, float), np.asarray(Y_train, float))
    return model

beta_shared = Y_coef.to_numpy(dtype=float).mean(axis=0)

## Residual-field correction models per cluster

In [ ]:
poly = PolynomialFeatures(degree=3, include_bias=True)

def fit_cluster_residual_models(train_regime_ids):
    train_mask = coef_df["regime_id"].isin(train_regime_ids)
    X_train_meta = X_meta.loc[train_mask]
    Y_train_beta = Y_coef.loc[train_mask]
    global_beta_model = fit_global_beta_predictor(X_train_meta, Y_train_beta)

    cluster_models = {}
    for cl in sorted(coef_df.loc[train_mask, "cluster"].unique()):
        rows_X = []
        rows_y = []

        cl_regimes = coef_df.loc[train_mask & (coef_df["cluster"] == cl), "regime_id"].tolist()
        for rid in cl_regimes:
            sub = regime_subsets[rid].copy()
            beta_hat = global_beta_model.predict(np.asarray(X_meta.loc[coef_df["regime_id"] == rid], float))[0]
            g_global = predict_with_beta(sub, beta_hat)
            resid = sub["predicted_flow"].to_numpy(dtype=float) - g_global

            Phi = poly.fit_transform(sub[["residual", "condition_coord"]].to_numpy(dtype=float))
            rows_X.append(Phi)
            rows_y.append(resid)

        Xc = np.vstack(rows_X)
        yc = np.concatenate(rows_y)

        model = Ridge(alpha=1.0)
        model.fit(Xc, yc)
        cluster_models[cl] = model

    clf = LogisticRegression(max_iter=2000, multi_class="auto")
    clf.fit(np.asarray(X_train_meta, float), coef_df.loc[train_mask, "cluster"].to_numpy())

    return global_beta_model, cluster_models, clf

def build_field_functions(beta_hat, cluster_models, clf, x_meta_row):
    proba = clf.predict_proba(np.asarray(x_meta_row, float))[0]
    class_labels = clf.classes_

    hard_cluster = int(class_labels[np.argmax(proba)])

    def global_field(r, c):
        x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
        return float(x @ beta_hat)

    def hard_corrected_field(r, c):
        phi = poly.fit_transform(np.array([[r, c]], dtype=float))
        delta = float(cluster_models[hard_cluster].predict(phi)[0])
        return float(global_field(r, c) + delta)

    weight_map = {int(cl): float(p) for cl, p in zip(class_labels, proba)}

    def soft_corrected_field(r, c):
        phi = poly.fit_transform(np.array([[r, c]], dtype=float))
        delta = 0.0
        for cl, model in cluster_models.items():
            delta += weight_map.get(int(cl), 0.0) * float(model.predict(phi)[0])
        return float(global_field(r, c) + delta)

    return global_field, hard_corrected_field, soft_corrected_field, hard_cluster, weight_map

## Compare global template vs residual-corrected models on harder blocks

In [ ]:
hard_block_masks = {
    "k_extrapolate_high": coef_df["k"].astype(float) == 7.0,
    "k_extrapolate_low": coef_df["k"].astype(float) == 3.0,
    "system_holdout::entropy": coef_df["system"].astype(str) == "entropy",
    "system_holdout::unevenness": coef_df["system"].astype(str) == "unevenness",
}

rows = []

for block_name, test_mask in hard_block_masks.items():
    train_mask = ~test_mask
    train_regimes = coef_df.loc[train_mask, "regime_id"].tolist()
    test_regimes = coef_df.loc[test_mask, "regime_id"].tolist()

    global_beta_model, cluster_models, clf = fit_cluster_residual_models(train_regimes)

    X_train_meta = X_meta.loc[train_mask]
    train_cluster_acc = accuracy_score(
        coef_df.loc[train_mask, "cluster"].to_numpy(),
        clf.predict(np.asarray(X_train_meta, float))
    )

    for rid in test_regimes:
        sub = regime_subsets[rid].copy()
        x_meta_row = X_meta.loc[coef_df["regime_id"] == rid]
        beta_true = Y_coef.loc[coef_df["regime_id"] == rid].to_numpy(dtype=float)[0]
        beta_hat = global_beta_model.predict(np.asarray(x_meta_row, float))[0]

        global_field, hard_field, soft_field, hard_cluster, weight_map = build_field_functions(
            beta_hat, cluster_models, clf, x_meta_row
        )

        def true_field(r, c):
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            return float(x @ beta_true)

        y_emp = sub["predicted_flow"].to_numpy(dtype=float)
        g_global = np.array([global_field(r, c) for r, c in zip(sub["residual"], sub["condition_coord"])], dtype=float)
        g_hard = np.array([hard_field(r, c) for r, c in zip(sub["residual"], sub["condition_coord"])], dtype=float)
        g_soft = np.array([soft_field(r, c) for r, c in zip(sub["residual"], sub["condition_coord"])], dtype=float)

        rows.append({
            "block": block_name,
            "regime_id": rid,
            "train_cluster_acc": train_cluster_acc,
            "pred_cluster_hard": hard_cluster,
            "soft_gate_weights": str(weight_map),

            "field_rmse_global": math.sqrt(mean_squared_error(y_emp, g_global)),
            "field_rmse_hard": math.sqrt(mean_squared_error(y_emp, g_hard)),
            "field_rmse_soft": math.sqrt(mean_squared_error(y_emp, g_soft)),

            "traj_rmse_global": trajectory_gap_from_field(sub, true_field, global_field),
            "traj_rmse_hard": trajectory_gap_from_field(sub, true_field, hard_field),
            "traj_rmse_soft": trajectory_gap_from_field(sub, true_field, soft_field),
        })

compare_df = pd.DataFrame(rows)
display(compare_df.head())

In [ ]:
summary_df = compare_df.groupby("block")[[
    "field_rmse_global", "field_rmse_hard", "field_rmse_soft",
    "traj_rmse_global", "traj_rmse_hard", "traj_rmse_soft",
    "train_cluster_acc"
]].mean().reset_index()

display(summary_df)

In [ ]:
# RMSE comparison by block

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(summary_df))
w = 0.26

axes[0].bar(x - w, summary_df["field_rmse_global"], width=w, label="global")
axes[0].bar(x, summary_df["field_rmse_hard"], width=w, label="hard-corrected")
axes[0].bar(x + w, summary_df["field_rmse_soft"], width=w, label="soft-corrected")
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[0].set_title("Field RMSE by harder block")
axes[0].legend()

axes[1].bar(x - w, summary_df["traj_rmse_global"], width=w, label="global")
axes[1].bar(x, summary_df["traj_rmse_hard"], width=w, label="hard-corrected")
axes[1].bar(x + w, summary_df["traj_rmse_soft"], width=w, label="soft-corrected")
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary_df["block"], rotation=30, ha="right")
axes[1].set_title("Trajectory RMSE by harder block")
axes[1].legend()

plt.tight_layout()
plt.show()

## Cluster prediction behavior

In [ ]:
cluster_counts = compare_df.groupby(["block", "pred_cluster_hard"]).size().reset_index(name="count")
display(cluster_counts)

pivot = cluster_counts.pivot(index="pred_cluster_hard", columns="block", values="count").fillna(0)
pivot.plot(kind="bar", figsize=(10, 5))
plt.ylabel("count")
plt.title("Hard-corrected cluster predictions by harder block")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
clf_full = LogisticRegression(max_iter=2000, multi_class="auto")
clf_full.fit(np.asarray(X_meta, float), coef_df["cluster"].to_numpy())
pred_full = clf_full.predict(np.asarray(X_meta, float))
cm = confusion_matrix(coef_df["cluster"].to_numpy(), pred_full)

print("Full-data cluster classification accuracy:", accuracy_score(coef_df["cluster"].to_numpy(), pred_full))
display(pd.DataFrame(cm))

## Representative regime comparison

In [ ]:
rep_idx = int(np.argmax(compare_df["traj_rmse_global"] - np.minimum(compare_df["traj_rmse_hard"], compare_df["traj_rmse_soft"])))
rep = compare_df.iloc[rep_idx]
regime_id = rep["regime_id"]

test_mask = coef_df["regime_id"] == regime_id
train_mask = ~test_mask

train_regimes = coef_df.loc[train_mask, "regime_id"].tolist()
global_beta_model, cluster_models, clf = fit_cluster_residual_models(train_regimes)

x_meta_row = X_meta.loc[coef_df["regime_id"] == regime_id]
beta_true = Y_coef.loc[coef_df["regime_id"] == regime_id].to_numpy(dtype=float)[0]
beta_hat = global_beta_model.predict(np.asarray(x_meta_row, float))[0]

global_field, hard_field, soft_field, hard_cluster, weight_map = build_field_functions(
    beta_hat, cluster_models, clf, x_meta_row
)

def true_field(r, c):
    x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
    return float(x @ beta_true)

sub = regime_subsets[regime_id]
cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
rmin, rmax = sub["residual"].min(), sub["residual"].max()
cgrid = np.linspace(cmin, cmax, 45)
r0s = np.linspace(np.quantile(sub["residual"], 0.1), np.quantile(sub["residual"], 0.9), 8)
flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

def integrate_field(field_fn, r0):
    r = float(np.clip(r0, rmin, rmax))
    traj = [r]
    for i in range(len(cgrid) - 1):
        c = cgrid[i]
        dc = float(cgrid[i+1] - cgrid[i])
        g = float(np.clip(field_fn(r, c), -flow_cap, flow_cap))
        r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
        traj.append(r)
    return np.array(traj)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
titles = ["Global", f"Hard-corrected (cluster {hard_cluster})", "Soft-corrected", "Regime-specific"]
field_fns = [global_field, hard_field, soft_field, true_field]

for ax, title, field_fn in zip(axes, titles, field_fns):
    for r0 in r0s:
        ax.plot(cgrid, integrate_field(field_fn, r0))
    ax.set_title(title)
    ax.set_xlabel("condition coordinate c")
axes[0].set_ylabel("residual")
fig.suptitle(f"Representative regime: {regime_id}", y=1.03)
plt.tight_layout()
plt.show()

print("Soft gate weights:", weight_map)

## Final verdicts

In [ ]:
def verdict_row(row):
    best_corr = min(row["traj_rmse_hard"], row["traj_rmse_soft"])
    if best_corr < row["traj_rmse_global"]:
        if row["traj_rmse_soft"] <= row["traj_rmse_hard"]:
            return "soft residual correction best"
        return "hard residual correction best"
    return "global template remains best"

summary_df["verdict"] = summary_df.apply(verdict_row, axis=1)
display(summary_df)

## Working conclusion

Notebook 55 treats regime structure as a correction field layered on top of the global template,
rather than only as coefficient variation.

A strong result is:
- harder blocks improve under residual-field corrections
- cluster-conditioned residuals explain structured mismatches
- soft gating can smooth transitions across nearby regimes

If that pattern holds on your real exports, the next notebook should be:

**Notebook 56 — joint coefficient + residual correction with adaptive gating**